In [55]:
import torch
import torch.nn as nn
import pandas as pd
import re

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

pd.set_option("display.max_colwidth", None)

Using device: cuda


In [56]:
VOCAB_PATH = r"C:\Users\nihal\Desktop\NLP_Project\bigru_vocab.pt"
vocab = torch.load(VOCAB_PATH)

MAX_LEN = 60

C:\Users\nihal\AppData\Local\Temp\ipykernel_22992\658094554.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vocab = torch.load(VOCAB_PATH)


In [57]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

def text_to_tensor(text):
    tokens = tokenize(text)
    ids = [vocab.get(t, 0) for t in tokens][:MAX_LEN]
    ids += [0] * (MAX_LEN - len(ids))
    return torch.tensor(ids).unsqueeze(0).to(DEVICE)

In [58]:
class BiGRURanker(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 200)
        self.embed_dropout = nn.Dropout(0.3)

        self.gru = nn.GRU(
            200,
            256,
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3
        )

        self.layer_norm = nn.LayerNorm(512)

        self.fc = nn.Sequential(
            nn.Linear(512 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 128),
            nn.ReLU(),

            nn.Linear(128, 1)
        )

    def encode(self, x):
        emb = self.embedding(x)
        emb = self.embed_dropout(emb)

        _, h = self.gru(emb)

        fwd = h[-2]
        bwd = h[-1]

        rep = torch.cat([fwd, bwd], dim=1)

        return self.layer_norm(rep)

    def forward(self, q, p):
        qv = self.encode(q)
        pv = self.encode(p)

        features = torch.cat([
            qv,
            pv,
            torch.abs(qv - pv),
            qv * pv
        ], dim=1)

        return self.fc(features).squeeze()

In [59]:
model = BiGRURanker(len(vocab)).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("BiGRU model loaded successfully")

C:\Users\nihal\AppData\Local\Temp\ipykernel_22992\3082149963.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=DE

BiGRU model loaded successfully


In [60]:
import pandas as pd
import torch
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BM25_PATH = r"C:\Users\nihal\Desktop\NLP_Project\bm25_subset_candidates.csv"

bm25_df = pd.read_csv(BM25_PATH)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", None)

print("BM25 loaded:", len(bm25_df))

BM25 loaded: 6000000


In [61]:
import re

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

def encode_text(text, vocab, max_len=60):
    tokens = tokenize(text)
    ids = [vocab.get(tok, 1) for tok in tokens]  # 1 = UNK
    ids = ids[:max_len]
    ids += [0] * (max_len - len(ids))            # padding
    return torch.tensor(ids)

In [62]:
example_query = "what does an endocrinologist do"

candidates = bm25_df[bm25_df["query"] == example_query].sort_values("rank")

top_candidates = candidates.head(10).copy()

display(top_candidates[["rank", "passage"]])

,rank,passage
11200,1,"What is an endocrinologist? An endocrinologist is a specially trained doctor who can diagnose and treat diseases that affect your glands, hormones and your endocrine system. The pancreas is part of the endocrine system, and insulin is one of the central hormones the body needs to function properly."
11201,2,"Your regular gynecologist may do some basic testing. Or, you may be referred to a reproductive endocrinologist (a doctor specializing in fertility) or a urologist (for male infertility) for more thorough fertility testing. Fertility testing involves both partners."
11202,3,"What does an Advertising Manager do? An advertising manager will typically do the following: Work with department heads or staff to discuss topics such as contracts, selection of advertising media, or products to be advertised. Gather and organize information to plan advertising campaigns."
11203,4,"An endocrinologist is a doctor who has studied the endocrine system and its diseases. These doctors know how to diagnose the diseases of the endocrine glands, and also how to treat them."
11204,5,"What Does A Logo Do. Just what does a logo do for your website? It gives visitors the first impression of your website. Ideally, your logo will represent the essence of your site, whether it's classy, serious, business-like, or off-beat. A website logo also helps contribute a consistent look and feel throughout all the pages of a website. An important consideration when building your website is the design of your logo."
11205,6,"college Video What DMT Does To Your Body What DMT Does To Your Body 1:21 What DMT Does To Your Body DMT is a psychedelic drug that can effect your body in strange and scary ways. Check out this video to learn what DMT does to your body. Transcript: DMT creates an almost immediate sense of euphoria, a feeling of sheer relaxation, and peaceful hallucinations.... DMT creates an almost immediate sense of euphoria, a feeling of sheer relaxation, and peaceful hallucinations."
11206,7,"What does suv stand for? Car keys. Suv meaning, definition, what is suv abbreviation for sport utility vehicle a large car with an engine that supplies power to. What does suv stand for in car? All acronyms dictionarycrossover what's the difference? Auto trader. Vwhat does suv stand for? I know it's a type of car. Suv definition by acronymfinder."
11207,8,"What Does Name Jashelle Mean You are honest, benevolent, brilliant and often inventive, full of high inspirations. You are courageous, honest, determined, original and creative. You are a leader, especially for a cause. Sometimes you do not care to finish what you start, and may leave details to others."
11208,9,"WHAT DOES IEP STAND FOR? WHAT DOES DIS MEAN? IEP is an acronym for Individualized Education Program, which is a unique document required by the government that aids the ability of a disabled student to receive a quality education."
11209,10,"Report Abuse. APO stands for Army Post Office. The AE stands for Army Europe and the number are just a place state side that the mail will be shipped from to get to the location overseas. It does it this way so it does not cost as much to ship things then it would with regular overseas mail.E: What does APO, AE 09356 mean? what does it mean? i know its a military address but what does AE stand for?"


In [63]:
scores = []

with torch.no_grad():
    for _, row in top_candidates.iterrows():
        q = encode_text(row["query"], vocab).unsqueeze(0).to(DEVICE)
        p = encode_text(row["passage"], vocab).unsqueeze(0).to(DEVICE)

        score = model(q, p)
        scores.append(score.item())

top_candidates["bigru_score"] = scores

In [64]:
print("BiGRU re-ranking:")
display(
    top_candidates
    .sort_values("bigru_score", ascending=False)
    [["bigru_score", "passage"]]
)

BiGRU re-ranking:


,bigru_score,passage
11200,6.872424,"What is an endocrinologist? An endocrinologist is a specially trained doctor who can diagnose and treat diseases that affect your glands, hormones and your endocrine system. The pancreas is part of the endocrine system, and insulin is one of the central hormones the body needs to function properly."
11203,4.726913,"An endocrinologist is a doctor who has studied the endocrine system and its diseases. These doctors know how to diagnose the diseases of the endocrine glands, and also how to treat them."
11201,2.044476,"Your regular gynecologist may do some basic testing. Or, you may be referred to a reproductive endocrinologist (a doctor specializing in fertility) or a urologist (for male infertility) for more thorough fertility testing. Fertility testing involves both partners."
11202,-2.160338,"What does an Advertising Manager do? An advertising manager will typically do the following: Work with department heads or staff to discuss topics such as contracts, selection of advertising media, or products to be advertised. Gather and organize information to plan advertising campaigns."
11204,-5.684857,"What Does A Logo Do. Just what does a logo do for your website? It gives visitors the first impression of your website. Ideally, your logo will represent the essence of your site, whether it's classy, serious, business-like, or off-beat. A website logo also helps contribute a consistent look and feel throughout all the pages of a website. An important consideration when building your website is the design of your logo."
11209,-11.192985,"Report Abuse. APO stands for Army Post Office. The AE stands for Army Europe and the number are just a place state side that the mail will be shipped from to get to the location overseas. It does it this way so it does not cost as much to ship things then it would with regular overseas mail.E: What does APO, AE 09356 mean? what does it mean? i know its a military address but what does AE stand for?"
11207,-11.604967,"What Does Name Jashelle Mean You are honest, benevolent, brilliant and often inventive, full of high inspirations. You are courageous, honest, determined, original and creative. You are a leader, especially for a cause. Sometimes you do not care to finish what you start, and may leave details to others."
11205,-12.121037,"college Video What DMT Does To Your Body What DMT Does To Your Body 1:21 What DMT Does To Your Body DMT is a psychedelic drug that can effect your body in strange and scary ways. Check out this video to learn what DMT does to your body. Transcript: DMT creates an almost immediate sense of euphoria, a feeling of sheer relaxation, and peaceful hallucinations.... DMT creates an almost immediate sense of euphoria, a feeling of sheer relaxation, and peaceful hallucinations."
11206,-14.646879,"What does suv stand for? Car keys. Suv meaning, definition, what is suv abbreviation for sport utility vehicle a large car with an engine that supplies power to. What does suv stand for in car? All acronyms dictionarycrossover what's the difference? Auto trader. Vwhat does suv stand for? I know it's a type of car. Suv definition by acronymfinder."
11208,-14.722222,"WHAT DOES IEP STAND FOR? WHAT DOES DIS MEAN? IEP is an acronym for Individualized Education Program, which is a unique document required by the government that aids the ability of a disabled student to receive a quality education."
